In [2]:
import pandas as pd

raw = pd.read_csv("State_to_State_Migration_Table_2022.csv", header=None)

# state names I want to keep
states = [
    "Alabama","Alaska","Arizona","Arkansas","California","Colorado","Connecticut",
    "Delaware","District of Columbia","Florida","Georgia","Hawaii","Idaho","Illinois",
    "Indiana","Iowa","Kansas","Kentucky","Louisiana","Maine","Maryland","Massachusetts",
    "Michigan","Minnesota","Mississippi","Missouri","Montana","Nebraska","Nevada",
    "New Hampshire","New Jersey","New Mexico","New York","North Carolina","North Dakota",
    "Ohio","Oklahoma","Oregon","Pennsylvania","Rhode Island","South Carolina",
    "South Dakota","Tennessee","Texas","Utah","Vermont","Virginia","Washington",
    "West Virginia","Wisconsin","Wyoming"
]

# row 6 has state names, row 7 has Estimate / MOE
state_row = raw.iloc[6]
type_row = raw.iloc[7]

# keep only columns where the header is a state name and the type is Estimate
keep_cols = [
    c for c in raw.columns
    if pd.notna(state_row[c]) and str(state_row[c]).strip() in states and str(type_row[c]).strip() == "Estimate"
]

# rows 9 onward contain the data block
df = raw.iloc[9:, [0] + keep_cols].copy()
df.columns = ["destination"] + [str(state_row[c]).strip() for c in keep_cols]

# keep only actual destination states
df["destination"] = df["destination"].astype(str).str.strip()
df = df[df["destination"].isin(states)]

# convert from wide to long
df_long = df.melt(
    id_vars="destination",
    var_name="origin",
    value_name="flow"
)

# clean flow values
df_long["flow"] = (
    df_long["flow"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df_long = df_long[df_long["flow"].str.match(r"^\d+$", na=False)]
df_long["flow"] = df_long["flow"].astype(int)

df_long.head(20)

,destination,origin,flow
1,Alaska,Alabama,151
2,Arizona,Alabama,727
3,Arkansas,Alabama,1911
4,California,Alabama,5204
5,Colorado,Alabama,700
6,Connecticut,Alabama,1005
7,Delaware,Alabama,203
8,District of Columbia,Alabama,419
9,Florida,Alabama,14734
10,Georgia,Alabama,21031


In [3]:
df_long.shape

(2550, 3)

In [4]:
df_long.sample(10)

,destination,origin,flow
1484,Colorado,New Hampshire,975
1464,Oklahoma,Nevada,1989
64,Illinois,Alaska,434
1381,California,Nebraska,3047
1026,Connecticut,Maryland,2460
2265,Massachusetts,Utah,1491
706,Texas,Illinois,25272
1254,New Jersey,Mississippi,0
264,Florida,Colorado,20980
1975,Oregon,Pennsylvania,624


In [5]:
import glob

files = glob.glob("State_to_State_Migration*.csv")

all_data = []

for file in files:
    year = file.split("_")[-1].split(".")[0]  # 把年份抓出来

    raw = pd.read_csv(file, header=None)

    state_row = raw.iloc[6]
    type_row = raw.iloc[7]

    keep_cols = [
        c for c in raw.columns
        if pd.notna(state_row[c]) and str(state_row[c]).strip() in states and str(type_row[c]).strip() == "Estimate"
    ]

    df = raw.iloc[9:, [0] + keep_cols].copy()
    df.columns = ["destination"] + [str(state_row[c]).strip() for c in keep_cols]

    df["destination"] = df["destination"].astype(str).str.strip()
    df = df[df["destination"].isin(states)]

    df_long = df.melt(
        id_vars="destination",
        var_name="origin",
        value_name="flow"
    )

    df_long["flow"] = (
        df_long["flow"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
    )

    df_long = df_long[df_long["flow"].str.match(r"^\d+$", na=False)]
    df_long["flow"] = df_long["flow"].astype(int)

    df_long["year"] = year

    all_data.append(df_long)

df_all = pd.concat(all_data, ignore_index=True)

df_all.sample(10)

,destination,origin,flow,year
12519,Maine,Virginia,307,2019
11887,Pennsylvania,North Carolina,7221,2019
8093,Utah,District of Columbia,149,2018
230,New Mexico,California,8897,2022
19954,California,Tennessee,8781,2015
576,Nebraska,Hawaii,174,2022
2862,Illinois,Connecticut,863,2023
14338,Rhode Island,New Mexico,77,2017
14269,Maine,New Jersey,662,2017
17880,New Mexico,Alabama,356,2015


In [6]:
files

['State_to_State_Migration_Table_2022.csv',
 'State_to_State_Migration_Table_2023.csv',
 'State_to_State_Migrations_Table_2021.csv',
 'State_to_State_Migration_Table_2024.csv',
 'State_to_State_Migrations_Table_2018.csv',
 'State_to_State_Migrations_Table_2019.csv',
 'State_to_State_Migrations_Table_2017.csv',
 'State_to_State_Migrations_Table_2016.csv',
 'State_to_State_Migrations_Table_2015.csv']

In [7]:
df_all["year"].unique()

array(['2022', '2023', '2021', '2018', '2019', '2017', '2016', '2015'],
      dtype=object)

In [8]:
# 计算每个州总流入
inflow = df_all.groupby(["year","destination"])["flow"].sum().reset_index()
inflow = inflow.rename(columns={"flow":"inflow"})

# 计算每个州总流出
outflow = df_all.groupby(["year","origin"])["flow"].sum().reset_index()
outflow = outflow.rename(columns={"origin":"destination","flow":"outflow"})

# merge
net = pd.merge(inflow, outflow, on=["year","destination"])

# net migration
net["net_migration"] = net["inflow"] - net["outflow"]

net.head()

,year,destination,inflow,outflow,net_migration
0,2015,Alabama,114876,100809,14067
1,2015,Alaska,34696,40552,-5856
2,2015,Arizona,253281,184476,68805
3,2015,Arkansas,81594,76673,4921
4,2015,California,514477,643710,-129233


In [9]:
net.sort_values("net_migration", ascending=False).head(10)

,year,destination,inflow,outflow,net_migration
315,2022,Florida,738969,489905,249064
264,2021,Florida,674740,469577,205163
349,2022,Texas,668338,494077,174261
60,2016,Florida,605018,433452,171566
213,2019,Florida,601611,457301,144310
298,2021,Texas,591395,447363,144032
9,2015,Florida,584938,445320,139618
400,2023,Texas,611942,478570,133372
366,2023,Florida,636933,510925,126008
111,2017,Florida,566476,447586,118890


In [10]:
net.sort_values("net_migration").head(10)

,year,destination,inflow,outflow,net_migration
259,2021,California,433402,841065,-407663
310,2022,California,475803,817669,-341866
287,2021,New York,287249,571041,-283792
361,2023,California,422075,690127,-268052
338,2022,New York,301461,545598,-244137
185,2018,New York,254447,458014,-203567
32,2015,New York,257611,448855,-191244
157,2018,California,501023,691145,-190122
83,2016,New York,260723,450136,-189413
236,2019,New York,254806,439708,-184902


In [11]:
df_unemp = pd.read_csv("staadata.csv")
df_unemp.head()
df_unemp.columns

Index(['States and selected areas:  Employment status of the civilian noninstitutional population,',
       'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5',
       'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9'],
      dtype='object')

In [16]:
df_unemp = pd.read_csv("staadata.csv", header=None)
df_unemp.iloc[:15, :10]

,0,1,2,3,4,5,6,7,8,9
0,States and selected areas: Employment status ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1976 to 2025(1) annual averages,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,FIPS Code,State and area,Year,Civilian non-institutional population,Civilian labor force,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,Total,Percent of population,Employment,NaN,Unemployment,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,Total,Percent of population,Total,Rate
6,NaN,NaN,NaN,NaN,NaN,NaN,Labor force,NaN,level,rate
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,01,Alabama,1976,"2,632,667","1,500,615",57.0,"1,399,921",53.2,"100,694",6.7
9,02,Alaska,1976,"239,917","164,450",68.5,"151,844",63.3,"12,606",7.7


In [17]:
unemp_raw = pd.read_csv("staadata.csv", header=None)

unemp = unemp_raw.iloc[8:, [1, 2, 9]].copy()
unemp.columns = ["state", "year", "unemployment_rate"]

unemp = unemp.dropna()
unemp["year"] = unemp["year"].astype(str)
unemp["unemployment_rate"] = pd.to_numeric(unemp["unemployment_rate"], errors="coerce")

# keep the same years as migration
unemp = unemp[unemp["year"].isin(df_all["year"].unique())]

# keep only states that appear in migration data
unemp = unemp[unemp["state"].isin(net["destination"].unique())]

unemp.head()

,state,year,unemployment_rate
2075,Alabama,2015,6.1
2076,Alaska,2015,6.3
2077,Arizona,2015,6.1
2078,Arkansas,2015,5.0
2079,California,2015,6.2


In [18]:
final_df = net.merge(
    unemp,
    left_on=["year", "destination"],
    right_on=["year", "state"],
    how="left"
)

final_df.head()

,year,destination,inflow,outflow,net_migration,state,unemployment_rate
0,2015,Alabama,114876,100809,14067,Alabama,6.1
1,2015,Alaska,34696,40552,-5856,Alaska,6.3
2,2015,Arizona,253281,184476,68805,Arizona,6.1
3,2015,Arkansas,81594,76673,4921,Arkansas,5.0
4,2015,California,514477,643710,-129233,California,6.2


In [19]:
final_df[["year", "destination", "net_migration", "unemployment_rate"]].head()
final_df["unemployment_rate"].isna().sum()

np.int64(0)

In [21]:
import plotly.express as px

# make sure year is numeric for plotting
final_df["year"] = final_df["year"].astype(int)

1. Top net migration gain/loss, 2023

In [22]:
df_2023 = final_df[final_df["year"] == 2023].copy()

top_gain = df_2023.sort_values("net_migration", ascending=False).head(10)
top_loss = df_2023.sort_values("net_migration", ascending=True).head(10)

In [23]:
fig = px.bar(
    top_gain,
    x="net_migration",
    y="destination",
    orientation="h",
    title="Top 10 Net Migration Gain States, 2023",
    labels={"net_migration": "Net Migration", "destination": "State"}
)

fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

In [24]:
fig = px.bar(
    top_loss,
    x="net_migration",
    y="destination",
    orientation="h",
    title="Top 10 Net Migration Loss States, 2023",
    labels={"net_migration": "Net Migration", "destination": "State"}
)

fig.update_layout(yaxis={"categoryorder": "total descending"})
fig.show()

2. Trend line for key states

In [25]:
states_selected = ["California", "Texas", "Florida", "New York"]

trend_df = final_df[final_df["destination"].isin(states_selected)].copy()

fig = px.line(
    trend_df,
    x="year",
    y="net_migration",
    color="destination",
    markers=True,
    title="Net Migration Over Time",
    labels={"year": "Year", "net_migration": "Net Migration", "destination": "State"}
)

fig.show()

3. Migration vs unemployment, 2023

In [26]:
fig = px.scatter(
    df_2023,
    x="unemployment_rate",
    y="net_migration",
    hover_name="destination",
    title="Migration vs Unemployment, 2023",
    labels={
        "unemployment_rate": "Unemployment Rate",
        "net_migration": "Net Migration"
    },
    trendline="ols"
)

fig.show()

In [29]:
pip install geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 2.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 6.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 8.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [geopandas]/4 [geopandas]
Note: you may need to restart the kernel to use updated packages.


In [31]:
import geopandas as gpd
import plotly.express as px

# read shapefile
states_map = gpd.read_file("tl_2025_us_state.shp")

# keep only states in our migration data
states_map = states_map[states_map["NAME"].isin(final_df["destination"].unique())].copy()

# merge all years
map_all = states_map.merge(
    final_df,
    left_on="NAME",
    right_on="destination",
    how="left"
)

# make sure year is string for animation
map_all["year"] = map_all["year"].astype(str)

# interactive animated map
fig = px.choropleth(
    map_all,
    locations="STUSPS",              # state abbreviation from shapefile
    locationmode="USA-states",
    color="net_migration",
    animation_frame="year",
    scope="usa",
    hover_name="NAME",
    hover_data={
        "net_migration": True,
        "inflow": True,
        "outflow": True,
        "unemployment_rate": True,
        "STUSPS": False
    },
    title="Net Migration by State Over Time"
)

fig.show()

In [33]:
final_df.to_csv("final_data.csv", index=False)
df_all.to_csv("migration_flows_clean.csv", index=False)